
# Machine Learning Pipeline for Imbalanced Multiclass Classification

## Background & Motivation

Real-world datasets often suffer from **class imbalance**, where some classes have significantly 
fewer samples than others. This creates several challenges:

1. **Biased Models**: Algorithms tend to favor majority classes, leading to poor minority class performance
2. **Misleading Accuracy**: High overall accuracy can mask terrible minority class predictions
3. **Business Impact**: Minority classes are often the most important (e.g., fraud detection, rare diseases)

## Common Approaches to Handle Imbalance

### 1. **Algorithmic Solutions**
- `class_weight='balanced'`: Penalizes misclassification of minority classes more heavily
- Cost-sensitive learning: Custom loss functions

### 2. **Resampling Techniques**
- **Under-sampling**: Remove majority class samples (e.g., RandomUnderSampler)
- **Over-sampling**: Duplicate/generate minority class samples (e.g., RandomOverSampler, SMOTE)
- **Hybrid**: Combine both approaches

### 3. **Evaluation Metrics for Imbalanced Data**
- **Avoid Accuracy**: Misleading on imbalanced data
- **Use Instead**: F1-score, Precision/Recall, ROC AUC, Balanced Accuracy
- **Multiclass**: Macro/weighted averages, One-vs-Rest (OVR) ROC AUC

## This Notebook's Approach

We'll systematically compare multiple strategies on a real imbalanced dataset:

1. **Dataset**: "thyroid-ann" from OpenML (3-class, ~3772 samples, imbalanced)
2. **Strategies**: (1)No resampling + class_weight; (2)Under-sampling; (3)Over-sampling; (4)SMOTE
3. **Pipeline**: Preprocessing → Resampling → Model (using imbalanced-learn)
4. **Evaluation**: Cross-validation + holdout test set with multiclass-appropriate metrics
5. **Model Comparison**: Test different algorithms on the same preprocessed data

## Key Technical Concepts

- **Pipeline Distinction**: sklearn vs imbalanced-learn pipelines (samplers support)
- **Stratified Splits**: Preserve class distribution in train/test splits
- **Cross-validation**: Robust strategy comparison without data leakage
- **deepcopy()**: Prevent object sharing across pipeline strategies


## 📝 Note: EDA & Feature Engineering Intentionally Skipped

This notebook **focuses on pipeline methodology** rather than exploratory data analysis or feature engineering. We're using a well-documented OpenML dataset ("thyroid-ann") and jumping directly to **imbalanced classification techniques**. 

In production workflows, you'd typically include:
- **Comprehensive EDA**: Feature distributions, correlations, missing values, outliers
- **Feature Engineering**: Domain-specific transformations, interaction terms, polynomial features


In [ ]:
# Import standard libraries

import pandas as pd
import numpy as np
from pathlib import Path
from copy import deepcopy

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from sklearn.pipeline import Pipeline as SkPipeline   # ✅ transformers/models only (no samplers)
from imblearn.pipeline import Pipeline as ImbPipeline  # ✅ supports samplers + transformers + models

# 1) WHY ALIASES?
# sklearn.pipeline.Pipeline CANNOT host imbalanced-learn samplers in the chain
# (it only knows "transformers" and "estimators"). 
# imbalanced-learn provides its own Pipeline that understands samplers (fit_resample).
# If you import both as"Pipeline" without aliasing, it's easy to accidentally use the sklearn one
# where you intended the imblearn one—your sampler steps would be ignored or
# raise errors. So we alias clearly to avoid mix-ups:


from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

import joblib

In [ ]:
# =========================================
# STEP 1. Prepare dataset
# - Fetch a real multiclass imbalanced dataset from OpenML
# - Automatically detect numeric/categorical columns
# - Extract X (features) and y (target)
# - Split early into train/test to mimic real-world:
#   training data can be resampled, but test must remain untouched and imbalanced
# =========================================

DATASET_NAME = "thyroid-ann"   # 3 classes, imbalanced, 3772 samples

# This line does TWO things at once:
# 1) Downloads the dataset from OpenML
# 2) Immediately separates features (X) and target (y)
X, y = fetch_openml(name=DATASET_NAME, version=1, as_frame=True, return_X_y=True)
#  ↑     ↑
#  |     └── Target variable (what we want to predict)
#  └── Features (input variables for prediction)


# Convert target to pandas categorical type (values unchanged, just metadata)
y = y.astype("category")
# WHY convert to "category" dtype?
# - Numeric targets (1,2,3) stay numeric but marked as discrete categories
# - String targets get memory-efficient categorical encoding
# - Prevents accidental arithmetic operations on class labels
# - Clear intent: Signals this is classification, not regression
# Note: sklearn models work with both numeric and categorical targets - this is just for data management


# =========================================
# Check class distribution to confirm imbalance
# =========================================
print("=== Class Distribution Analysis ===")
print(y.value_counts().sort_index())


# Auto-detect feature types
numeric_features = X.select_dtypes(include=["number", "float", "int"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number", "float", "int"]).columns.tolist()
# If dataset has no categoricals, that's fine—OneHot step will just handle an empty list.

# Define feature types
numeric_features = X.select_dtypes(include=["number", "float", "int"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number", "float", "int"]).columns.tolist()
# Note: 
# thyroid-ann dataset is all-numeric, categorical_features is just for demonstration
# If dataset has no categoricals, that's fine—OneHot step will just handle an empty list.
# OneHotEncoder is the safest default for unknown categorical data - makes no assumptions
# about category relationships or order, works well with most ML algorithms

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
# WHAT DOES stratify=y DO?
# It preserves the class distribution of y across the train/test split.
# On imbalanced data, this is crucial: both splits keep similar class ratios,
# preventing a "lucky" or "unlucky" split that would distort evaluation.

In [ ]:
# =========================================
# STEP 2. Define preprocessing transformers
# - Scale numeric features
# - OneHotEncode categorical features
# - Use sparse_output=False so SMOTE (which needs dense arrays) can work
# - Wrap them in a ColumnTransformer so preprocessing happens automatically inside pipeline
# =========================================
numeric_transformer = SkPipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = SkPipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# =========================================
# STEP 3. Define base model
# 3) WHAT PARAMETERS CAN WE SET HERE?
# Commonly tuned LogisticRegression params include:
# - penalty ("l2", "l1", "elasticnet", "none")
# - C (inverse regularization strength, float > 0)
# - solver ("liblinear", "lbfgs", "saga", etc.; depends on penalty)
# - class_weight (None or "balanced")
# - max_iter (iterations cap)
# - l1_ratio (for elasticnet, between 0 and 1)
# - random_state (for reproducibility where applicable)
# Here, for multiclass we typically use solver='lbfgs' and multi_class='multinomial'.
# We will still set class_weight later per strategy where applicable.
# =========================================
base_lr = LogisticRegression(max_iter=200, solver="lbfgs", multi_class="multinomial")

# For resampling strategies, this base configuration is fine:
# - max_iter=200: Sufficient iterations
# - solver="lbfgs": Good for multiclass
# - multi_class="multinomial": Proper multiclass handling
# - penalty="l2" (default): Reasonable regularization
# - C=1.0 (default): Balanced regularization strength

In [ ]:
# =========================================
# STEP 4. Build multiple pipelines (different resampling strategies)
# - Each pipeline = preprocessing + (optional resampling) + model
# - ImbPipeline ensures resampling only happens on training folds, never on validation/test
#
# 4) WHY deepcopy(...) IN STRATEGIES?
# We reuse the same preprocessor and base model objects across multiple pipelines.
# Pipelines mutate their steps during fit; without deepcopy, objects would be shared
# across strategies, causing cross-contamination (e.g., fitted encoders reused elsewhere).
# deepcopy gives each strategy its own clean, independent copy.


# =========================================
# Creating a dictionary where:
# - Keys are strategy names (strings)
# - Values are complete ML pipelines

# Each pipeline is a sequence of steps:
# ImbPipeline([
#     ("step_name", step_object),    # Step 1: Preprocessing
#     ("another_step", another_obj), # Step 2: Resampling (for imbalanced data)
#     ("final_step", model_obj)      # Step 3: Model
# ])

# =========================================
# L1 (Lasso): Eliminates weak features → coefficients become exactly 0
# L2 (Ridge): Shrinks all features → coefficients get smaller but never 0  
# C: Controls regularization strength for ALL penalty types (always important)
# ElasticNet - "Best of Both Worlds": Combines L1 and L2, balancing feature selection and coefficient shrinking. More complex, requires tuning both C and l1_ratio.
# l1_ratio: ONLY matters for ElasticNet (mixing L1 and L2)


# Algorithmic approach: Change model behavior
# Resampling approach: Change the training data

strategies = {
    # (A) No resampling. Using Algorithmic approach: Change model behavior

    # No penalty: Neither C nor l1_ratio matter
    "no_regularization": ImbPipeline([
        ("prep", deepcopy(preprocessor)),
        ("model", deepcopy(base_lr).set_params(
            class_weight="balanced",
            penalty=None,           # No regularization
            # C=100,                # ❌ IGNORED: No regularization applied
            # l1_ratio=0.5,         # ❌ IGNORED: No penalty to mix
            solver="lbfgs"          # Solver must support no penalty (lbfgs or saga)
        ))
    ]),

    # L1 penalty: C matters, l1_ratio is IGNORED  
    "l1_regularization": ImbPipeline([
        ("prep", deepcopy(preprocessor)),
        ("model", deepcopy(base_lr).set_params(
            class_weight="balanced", 
            penalty="l1",           # L1 penalty
            C=1.0,                  # Controls regularization strength
            # l1_ratio=0.8,         # ❌ IGNORED: Has no effect with L1
            solver="saga"           # Solver must support L1 and multinomial
        ))
    ]),

    # L2 penalty: C matters, l1_ratio is IGNORED
    "l2_regularization": ImbPipeline([
        ("prep", deepcopy(preprocessor)),
        ("model", deepcopy(base_lr).set_params(
            class_weight="balanced",
            penalty="l2",           # L2 penalty
            C=0.1,                  # Controls regularization strength
            # l1_ratio=0.5,         # ❌ IGNORED: Has no effect with L2
            solver="lbfgs"
        ))
    ]),

    # ElasticNet: BOTH C and l1_ratio matter
    "elasticnet_regularization": ImbPipeline([
        ("prep", deepcopy(preprocessor)),
        ("model", deepcopy(base_lr).set_params(
            class_weight="balanced",
            penalty="elasticnet",   # ElasticNet penalty
            C=0.5,                  # Overall regularization strength
            l1_ratio=0.3,           # 30% L1, 70% L2 mix
            solver="saga"           # Only solver supporting elasticnet
        ))
    ]),


    # (B) Random under-sampling. Resampling approach: Change the training data.
    "random_undersample": ImbPipeline([
        ("prep", deepcopy(preprocessor)),
        ("under", RandomUnderSampler(random_state=42)),
        ("model", deepcopy(base_lr))
    ]),

    # (C) Random over-sampling. Resampling approach: Change the training data.
    "random_oversample": ImbPipeline([
        ("prep", deepcopy(preprocessor)),
        ("over", RandomOverSampler(random_state=42)),
        ("model", deepcopy(base_lr))
    ]),

    # (D) SMOTE (synthetic over-sampling). Resampling approach: Change the training data.
    "smote": ImbPipeline([
        ("prep", deepcopy(preprocessor)),
        ("smote", SMOTE(random_state=42)),
        ("model", deepcopy(base_lr))
    ]),
}

In [ ]:
# =========================================
# STEP 5. Compare strategies with cross-validation
# - Use StratifiedKFold to keep class distribution balanced in folds
# - Evaluate multiple metrics suited for MULTICLASS:
#   * ROC AUC (OVR, weighted)  -> 'roc_auc_ovr_weighted'
#   * F1 (macro average)       -> 'f1_macro'
#   * Precision (macro)        -> 'precision_macro'
#   * Recall (macro)           -> 'recall_macro'
#   * Balanced Accuracy        -> 'balanced_accuracy' (already multiclass-aware)
# - Collect mean scores for each strategy and rank them
#
# 4b) WHAT'S THE SYNTAX OF cross_validate?
# cross_validate(estimator, X, y, cv=..., scoring=..., n_jobs=..., return_train_score=...)
# - estimator: a (pipeline) model with fit/predict(/predict_proba)
# - scoring: str or dict of metrics; with dict, you get keys like "test_roc_auc_ovr_weighted"
# - returns a dict of arrays (scores per fold). Take np.mean(...) to summarize.
# =========================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#                    ↑ This means "split data into 5 pieces"
scoring = {
    "roc_auc_ovr_weighted": "roc_auc_ovr_weighted",
    "f1_macro": "f1_macro",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "balanced_accuracy": "balanced_accuracy"
}

# What is n_splits=5?
# Think of cross-validation like testing your model 5 different times to make sure it's really good, not just lucky.
# Your training data gets split into 5 equal pieces:

# Round 1: Train on pieces 1,2,3,4 → Test on piece 5
# Round 2: Train on pieces 1,2,3,5 → Test on piece 4  
# Round 3: Train on pieces 1,2,4,5 → Test on piece 3
# Round 4: Train on pieces 1,3,4,5 → Test on piece 2
# Round 5: Train on pieces 2,3,4,5 → Test on piece 1

# Final result: Average performance across all 5 tests


cv_summary = []
for name, pipe in strategies.items():
    cv_res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    #                                                                         ↑ -1 means use ALL CPU cores
    cv_summary.append({
        "strategy": name,
        "roc_auc_mean": np.mean(cv_res["test_roc_auc_ovr_weighted"]),
        "f1_mean": np.mean(cv_res["test_f1_macro"]),
        "precision_mean": np.mean(cv_res["test_precision_macro"]),
        "recall_mean": np.mean(cv_res["test_recall_macro"]),
        "bal_acc_mean": np.mean(cv_res["test_balanced_accuracy"])
    })


# n_jobs=1 (sequential - one at a time,without parallelization):
# Fold 1: Train → Test → Score  (takes 30 seconds)
# Fold 2: Train → Test → Score  (takes 30 seconds)  
# Fold 3: Train → Test → Score  (takes 30 seconds)
# Fold 4: Train → Test → Score  (takes 30 seconds)
# Fold 5: Train → Test → Score  (takes 30 seconds)
# Total time: 150 seconds

# n_jobs=-1 (parallel - all at once, assuming 5+ CPU cores):
# Fold 1, 2, 3, 4, 5: All run simultaneously
# Total time: ~30 seconds (5x faster!)

# To see how many CPU cores your machine has, uncomment below:
# import os
# print(f"Your CPU has {os.cpu_count()} cores")
# If you set n_jobs=5 but only have 2 CPU cores, nothing breaks - but you won't get the speed you expect.



# Choose your ranking metric: ROC AUC (weighted OVR) is a good default for multiclass
cv_df = pd.DataFrame(cv_summary).sort_values("roc_auc_mean", ascending=False)

# Different ranking strategies for different business needs.
# Business scenario guide for metric selection:
# Medical diagnosis (don't miss sick patients):
# cv_df = pd.DataFrame(cv_summary).sort_values("recall_mean", ascending=False)

# Spam detection (don't block important emails): 
# cv_df = pd.DataFrame(cv_summary).sort_values("precision_mean", ascending=False)

# General balanced performance:
# cv_df = pd.DataFrame(cv_summary).sort_values("f1_mean", ascending=False)

# All classes equally important:
# cv_df = pd.DataFrame(cv_summary).sort_values("bal_acc_mean", ascending=False)

# Imbalanced multiclass (current choice):
# cv_df = pd.DataFrame(cv_summary).sort_values("roc_auc_mean", ascending=False)


print("=== CV Results (multiclass) ===")
print(cv_df.to_string(index=False))

best_name = cv_df.iloc[0]["strategy"]
best_pipe = strategies[best_name]
print(f"\nSelected strategy: {best_name}")


In [ ]:
# =========================================
# STEP 6. Fit best pipeline on full training set
# 5) WHY FIT AGAIN HERE?
# cross_validate trains multiple *fold-specific* clones and discards them.
# After choosing the best strategy, we must fit ONE final model on ALL training data
# to capture all available signal before evaluating/deploying.
# =========================================
best_pipe.fit(X_train, y_train)


In [ ]:
# =========================================
# STEP 7. Evaluate on untouched test set
# 6) WHY EVALUATE AGAIN—ISN'T CV ENOUGH?
# CV estimates generalization using *validation folds* from the training data.
# A final evaluation on the held-out test set (never touched during CV or resampling)
# is the gold-standard check against selection bias and overfitting from model/strategy choice.
# =========================================
y_proba = best_pipe.predict_proba(X_test)  # shape: (n_samples, n_classes)
y_pred = best_pipe.predict(X_test)

print("\n=== Test Set Results (multiclass) ===")
# Multiclass ROC AUC: use OVR with weighted average
print(f"ROC AUC (OVR, weighted): {roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

In [ ]:
# =========================================
# STEP 8. Save artifacts (model + preprocessor)
# 7) WHY SAVE PREPROCESSOR-ONLY AND ALSO THE FULL PIPELINE?
# - Full pipeline (preprocessor + model): easiest for deployment; pass raw features in,
#   get predictions out with identical transformations used in training.
# - Preprocessor-only: useful when training multiple models on the same frozen features,
#   or when you need to transform data once (offline) and store the design matrix.
# =========================================
Path("artifacts").mkdir(exist_ok=True)

# Full pipeline (recommended for deployment)
joblib.dump(best_pipe, "artifacts/mc_lr_pipeline.pkl")
print("Saved: artifacts/mc_lr_pipeline.pkl")

# Preprocessor only (optional)
fitted_preprocessor = deepcopy(preprocessor).fit(X_train, y_train)
joblib.dump(fitted_preprocessor, "artifacts/preprocessor_fitted.pkl")
print("Saved: artifacts/preprocessor_fitted.pkl")

In [ ]:
# =========================================
# Train another model using the saved preprocessor
# - Load preprocessor-only artifact
# - Transform raw X_train/X_test to design matrices
# - Train a different model (e.g., RandomForest) on the transformed data
# - Evaluate on transformed test set
# =========================================

from sklearn.ensemble import RandomForestClassifier

# Load the preprocessor you already saved
preprocessor = joblib.load("artifacts/preprocessor_fitted.pkl")

# Transform the train/test sets into numeric arrays
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Train a new model on the already-processed data
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_transformed, y_train)

# Predictions
y_proba = rf_model.predict_proba(X_test_transformed)   # probabilities for ROC AUC
y_pred = rf_model.predict(X_test_transformed)

# === Evaluation (same as full pipeline) ===
print("\n=== RandomForest Test Results ===")
# Multiclass ROC AUC (weighted OVR)
print(f"ROC AUC (OVR, weighted): {roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'):.4f}")

# Confusion Matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Classification Report (per class + macro/weighted averages)
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

In [ ]:

from sklearn.ensemble import HistGradientBoostingClassifier


# Load the preprocessor you already saved
preprocessor = joblib.load("artifacts/preprocessor_fitted.pkl")

# Transform the train/test sets into numeric arrays
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Train a Gradient Boosting model on the already-processed data
gb_model = HistGradientBoostingClassifier(random_state=42)
gb_model.fit(X_train_transformed, y_train)

# Predictions
y_proba = gb_model.predict_proba(X_test_transformed)   # probabilities for ROC AUC
y_pred = gb_model.predict(X_test_transformed)

# === Evaluation (same metrics as your pipeline) ===
print("\n=== Gradient Boosting (HGB) Test Results ===")
# Multiclass ROC AUC (OVR, weighted)
print(f"ROC AUC (OVR, weighted): {roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'):.4f}")

# Confusion Matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Classification Report (per class + macro/weighted averages)
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))


In [ ]:
# =========================================
# STEP 9. Deploy saved pipeline for real-world predictions
# - Load the full pipeline from disk
# - Make predictions on new data
# =========================================

import pandas as pd
import numpy as np
import joblib

# Load the saved pipeline
loaded_pipeline = joblib.load("artifacts/mc_lr_pipeline.pkl")
print("✅ Pipeline loaded successfully!")

# =========================================
# Create sample new data for deployment demonstration
# (In production, this would come from your application/database)
# =========================================


# For demonstration, let's use actual samples from test set (but pretend they're new data)
sample_data = X_test.iloc[:3].copy()  # Take first 3 samples from test set
print("Sample input data:")
print(sample_data)
print()

# =========================================
# Make predictions using the loaded pipeline
# =========================================

# Get class predictions
predictions = loaded_pipeline.predict(sample_data)
print("🎯 Class Predictions:")
print(f"Predicted classes: {predictions}")

# Get probability predictions  
probabilities = loaded_pipeline.predict_proba(sample_data)
print(f"\n📊 Prediction Probabilities:")
print(f"Shape: {probabilities.shape}")  # (n_samples, n_classes)

# Display probabilities in a nice format
class_names = loaded_pipeline.classes_  # Get class names from the pipeline
for i, (pred_class, probs) in enumerate(zip(predictions, probabilities)):
    print(f"\nSample {i+1}:")
    print(f"  Predicted Class: {pred_class}")
    print("  Class Probabilities:")
    for class_name, prob in zip(class_names, probs):
        print(f"    Class {class_name}: {prob:.4f} ({prob*100:.1f}%)")








In [ ]:
# =========================================
# Prediction function
# =========================================

def predict_new_data(pipeline_path, new_data):
    """
    Production function to make predictions on new data
    
    Parameters:
    -----------
    pipeline_path : str
        Path to saved pipeline (.pkl file)
    new_data : pd.DataFrame
        New data with same structure as training data
        
    Returns:
    --------
    dict : Dictionary containing predictions and probabilities
    """
    
    # Load pipeline
    pipeline = joblib.load(pipeline_path)
    
    # Make predictions
    predictions = pipeline.predict(new_data)
    probabilities = pipeline.predict_proba(new_data)
    
    # Format results
    results = {
        'predictions': predictions.tolist(),
        'probabilities': probabilities.tolist(),
        'class_names': pipeline.classes_.tolist()
    }
    
    return results

# Example usage of production function
print("\n" + "="*50)
print("🚀 DEPLOYMENT EXAMPLE")
print("="*50)

results = predict_new_data("artifacts/mc_lr_pipeline.pkl", sample_data)

print(f"New data predictions: {results['predictions']}")
print(f"Available classes: {results['class_names']}")

# Display formatted results
for i, (pred, probs) in enumerate(zip(results['predictions'], results['probabilities'])):
    print(f"\n📋 Sample {i+1} Results:")
    print(f"   Predicted: Class {pred}")
    print(f"   Confidence: {max(probs)*100:.1f}%")
    
    # Show top 2 most likely classes
    sorted_indices = np.argsort(probs)[::-1]  # Sort in descending order
    print("   Top predictions:")
    for j in range(min(2, len(probs))):  # Show top 2
        class_idx = sorted_indices[j]
        class_name = results['class_names'][class_idx]
        confidence = probs[class_idx] * 100
        print(f"     {j+1}. Class {class_name}: {confidence:.1f}%")